In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from urllib.parse import urljoin
import time

# An empty list to contain all the book data
all_books = []

headers = {                 # headers mimic real browser
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/146.0.0.0 Safari/537.36"
        }

try:
    for page in range(1,2):   # Scraping 10 pages (running a loop through maximum pages)
        url = f"https://books.toscrape.com/catalogue/page-{page}.html"     # urls for each page
        response = requests.get(url, headers=headers)    # the actual request that obtains the content from that webpage
        response.encoding = "utf-8"     # sets the encoding for windoes to read special symbols easily

        # By doing this, loop will stop if the pages do not exist further
        if response.status_code != 200:
            print(f"Failed on page {page}")
            break
        
        # Parsing the html
        soup = BeautifulSoup(response.text, "lxml")
        
        # 1. Targetting the 'article' tag
        books = soup.find_all("article", class_= "product_pod")

        for book in books:
            # Gets titles and prices of the books
            title = book.h3.a.get("title")
            price = book.find("p", class_="price_color").text
            price = float(price.replace("£", ""))

            # Gets the rating (Eg. Three)
            rating = book.find("p", class_="star-rating")["class"][1]

            # Gets stock availability with whitespace cleanup
            stock = book.find("p", class_="instock availability").text.strip()
            
            # Gets the URL of the books and create clickable links
            url = book.h3.a.get('href')
            full_url = urljoin("https://books.toscrape.com/catalogue/", url)
            
            # Append data as a dictionary to our list
            all_books.append({
                "Title": title,
                "Price": price,
                "Rating": rating,
                "Stock": stock,
                "URL": full_url
            })

        time.sleep(1)       # prevents sudden load of requests

    # Create a DataFrame using Pandas
    df = pd.DataFrame(all_books)

    # Export to csv (index=False prevents extra row numbers)
    df.to_csv("Scraped Datatest.csv", index=False, encoding="utf-8-sig")
    df.to_json("Scraped Datatest.json", index=False)
    print("Successfully Exported as .csv and .json!")
    print(df.head())

except Exception as e:      # in case any exception occurs, an error message will display
    print("An Error has occured!", e)

Successfully Exported as .csv !
                                   Title  Price Rating     Stock  \
0                   A Light in the Attic  51.77  Three  In stock   
1                     Tipping the Velvet  53.74    One  In stock   
2                             Soumission  50.10    One  In stock   
3                          Sharp Objects  47.82   Four  In stock   
4  Sapiens: A Brief History of Humankind  54.23   Five  In stock   

                                                 URL  
0  https://books.toscrape.com/catalogue/a-light-i...  
1  https://books.toscrape.com/catalogue/tipping-t...  
2  https://books.toscrape.com/catalogue/soumissio...  
3  https://books.toscrape.com/catalogue/sharp-obj...  
4  https://books.toscrape.com/catalogue/sapiens-a...  
